# Pilot Training of WIND and SpherIDiff

This notebook trains the models of the pilot experiment on Kaggle. The three WIND sizes run one after another on the first GPU, and SpherIDiff runs at the same time on the second GPU. With one GPU, all four runs follow each other. The notebook uses the ERA5 cache at 5.6 degrees, attached as a Kaggle input, and the repository code, cloned from GitHub, and it is designed for a background run, in which Kaggle executes the whole notebook and keeps the files of `/kaggle/working` when it ends.

Every run writes its step log, evaluation log, resumable checkpoint, EMA weights and, when it finishes all steps, a final checkpoint into `/kaggle/working/runs/<run name>`. Everything is packed into `pilot_results.zip` in the working folder. Checkpoints at 25%, 50% and 75% of the steps are kept for every run, so that a short learning-rate decay can be started from an early point.

In [ ]:
import os
import sys
import time
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
SESSION_HOURS = 9.0
PACKAGING_RESERVE_MINUTES = 20
WIND_EXPERIMENTS = ["pilot_wind_s", "pilot_wind_m", "pilot_wind_l"]
SPHERIDIFF_EXPERIMENTS = ["pilot_spheridiff_b4"]
ARCHIVE = WORK_ROOT / "pilot_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. The data cache is read from the attached dataset `thesyeddaniyal/era5-pilot-6degrees`, at `/kaggle/input/datasets/thesyeddaniyal/era5-pilot-6degrees`. The folder that holds `stats.npz` is used as the cache root. When that input path does not exist, the whole of `/kaggle/input` is searched for `stats.npz`.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
DATASET_ROOT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
CACHE_ROOT = locate("stats.npz", DATASET_ROOT if DATASET_ROOT.exists() else INPUT_ROOT)
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

Each job is a separate process with its own log file in `runs/_logs`, and the processes of different GPUs run at the same time. The notebook prints every line that a job writes as soon as it appears, prefixed with the elapsed time, the GPU and the job name, so that all jobs can be followed in one output. The same lines are kept in `runs/_logs/parallel.log`.

The jobs and the packing of the results share one cell with a `try` and a `finally`. When a job fails, or the cell is interrupted, the figures, the status file and the archive are still produced. The archive is also refreshed every 30 minutes during the run. The time limit of the session is set by `SESSION_HOURS`, and the jobs are told to stop with a checkpoint before it, leaving time for packing.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import assign_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

DEADLINE = time.time() + SESSION_HOURS * 3600 - PACKAGING_RESERVE_MINUTES * 60
GROUPS = [[train_job(name) for name in WIND_EXPERIMENTS], [train_job(name) for name in SPHERIDIFF_EXPERIMENTS]]
QUEUES = assign_queues(GROUPS, torch.cuda.device_count())
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", CACHE_ROOT, DEADLINE)

COMPARISONS = [(run, reference) for run in SPHERIDIFF_EXPERIMENTS for reference in WIND_EXPERIMENTS]

RUNNER = None
try:
    RUNNER = ParallelRunner(QUEUES, CONTEXT, refresh=lambda: refresh_archive(RUNS_ROOT, ARCHIVE))
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    finalise(RUNS_ROOT, ARCHIVE, OUTCOMES, WIND_EXPERIMENTS + SPHERIDIFF_EXPERIMENTS, references=WIND_EXPERIMENTS, comparisons=COMPARISONS)
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summaries list the steps, training time, speed and final loss of every run. The figures show the validation loss, the normalised forecast error and the normalised CRPS against steps, samples seen and training time, with the final value of each WIND size as a dashed line, and the training loss smoothed over 50 steps. The crossing of a SpherIDiff curve with a WIND line can be read from these figures. The comparison summary lists, for each SpherIDiff run against each WIND run, the step at which SpherIDiff reaches the final error of WIND and stays there, the errors at fixed fractions of training, the mean error over the whole curve, and a paired difference with a 95% interval over the evaluation windows.

In [ ]:
import json

for name in WIND_EXPERIMENTS + SPHERIDIFF_EXPERIMENTS:
    path = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(path.read_text()) if path.exists() else "no summary, see runs/_logs")

summary = RUNS_ROOT / "comparison_summary.txt"
if summary.exists():
    print(summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))